In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score, recall_score


In [2]:
import pandas as pd
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)
df.head(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
def fillna(df):
    df = df.copy()  # 명시적 복사
    df['Age'] = df['Age'].fillna(df['Age'].mean())
    df['Cabin'] = df['Cabin'].fillna('N')
    df['Embarked'] = df['Embarked'].fillna('N')
    df['Fare'] = df['Fare'].fillna(0)
    return df

def drop_features(df):
    return df.drop(['PassengerId', 'Name', 'Ticket'], axis=1)

def format_features(df):
    df = df.copy()
    df['Cabin'] = df['Cabin'].str[:1]
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        df[feature] = le.fit_transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수 호출
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

In [14]:
def get_clf_eval(y_test, pred):
    confusion = confusion_matrix(y_test, pred)
    accuracy = accuracy_score(y_test, pred)
    precision = precision_score(y_test, pred)
    recall = recall_score(y_test, pred)

    print("Confusion Matrix", confusion, sep='\n')
    print('*'*20)
    print('accuracy', 'precision', 'recall')
    print(accuracy, precision, recall)

In [5]:
titanic_df = transform_features(df)

In [6]:
y_titanic_df = titanic_df['Survived']
X_titanic_df = titanic_df.drop('Survived', axis=1)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_titanic_df,
                                                    y_titanic_df,
                                                    test_size=0.2,
                                                    random_state=0 )


# 성능비교 - 로지스틱회귀

In [15]:
#로지스틱회귀 분류모델 생성
from sklearn.linear_model import LogisticRegression

lr_clf = LogisticRegression(max_iter=2000)
lr_clf.fit(X_train, y_train)
pred = lr_clf.predict(X_test)

#정확도, 정밀도, 재현율
get_clf_eval(y_test, pred)

Confusion Matrix
[[92 18]
 [16 53]]
********************
accuracy precision recall
0.8100558659217877 0.7464788732394366 0.7681159420289855


# Custom 학습모델 만들기

In [10]:
from sklearn.base import BaseEstimator

import numpy as np

class MyDummyClassifier(BaseEstimator) :
    
    def fit(self, X, y) :
        pass
    
    def predict(self, X):
        pred = np.zeros((X.shape[0],1))
        for i in range(X.shape[0]):
            if X['Sex'].iloc[i] == 1:  # 남자면 모두 사망
                pred[i]=0
            else :
                pred[i]=1  # 여자면 모두 생존
        return pred


In [11]:
dummy_model = MyDummyClassifier()
dummy_model.fit(X_train, y_train)
pred = dummy_model.predict(X_test)

get_clf_eval(y_test, pred)

[[92 18]
 [20 49]]
********************
0.7877094972067039 0.7313432835820896 0.7101449275362319


# 랜덤포레스트, KNN 의 정밀도, 재현율 비교하기

In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

# 모델 생성 및 학습
rf_clf = RandomForestClassifier(n_estimators=100, random_state=0)
knn_clf = KNeighborsClassifier(n_neighbors=5)

rf_clf.fit(X_train, y_train)
knn_clf.fit(X_train, y_train)

# 예측
rf_pred = rf_clf.predict(X_test)
knn_pred = knn_clf.predict(X_test)

# 평가
print("=" * 40)
print("로지스틱 회귀")
print("=" * 40)
get_clf_eval(y_test, pred)

print("\n" + "=" * 40)
print("랜덤포레스트")
print("=" * 40)
get_clf_eval(y_test, rf_pred)

print("\n" + "=" * 40)
print("KNN")
print("=" * 40)
get_clf_eval(y_test, knn_pred)

로지스틱 회귀
Confusion Matrix
[[92 18]
 [16 53]]
********************
accuracy precision recall
0.8100558659217877 0.7464788732394366 0.7681159420289855

랜덤포레스트
Confusion Matrix
[[99 11]
 [20 49]]
********************
accuracy precision recall
0.8268156424581006 0.8166666666666667 0.7101449275362319

KNN
Confusion Matrix
[[94 16]
 [31 38]]
********************
accuracy precision recall
0.7374301675977654 0.7037037037037037 0.5507246376811594


# 분류모델의 임계치의 확인

In [26]:
pred_proba = lr_clf.predict_proba(X_test)
pred_proba



array([[0.85343168, 0.14656832],
       [0.89024994, 0.10975006],
       [0.92538153, 0.07461847],
       [0.0578497 , 0.9421503 ],
       [0.32222037, 0.67777963],
       [0.49139694, 0.50860306],
       [0.08752751, 0.91247249],
       [0.06753815, 0.93246185],
       [0.41894427, 0.58105573],
       [0.30525181, 0.69474819],
       [0.90908773, 0.09091227],
       [0.27547667, 0.72452333],
       [0.87926421, 0.12073579],
       [0.09296848, 0.90703152],
       [0.03689938, 0.96310062],
       [0.23636285, 0.76363715],
       [0.85867342, 0.14132658],
       [0.75480442, 0.24519558],
       [0.91048942, 0.08951058],
       [0.63997181, 0.36002819],
       [0.66515928, 0.33484072],
       [0.05646445, 0.94353555],
       [0.87926591, 0.12073409],
       [0.56497725, 0.43502275],
       [0.30306896, 0.69693104],
       [0.10943772, 0.89056228],
       [0.89947197, 0.10052803],
       [0.30363368, 0.69636632],
       [0.17273039, 0.82726961],
       [0.38010705, 0.61989295],
       [0.

In [17]:
pred_proba = lr_clf.predict_proba(X_test)
pos_proba = pred_proba[:, 1] #양성일 확률

threshold = 0.4
custom_proba = (pos_proba >= threshold).astype(int)
confusion_matrix(y_test, custom_proba)

array([[86, 24],
       [13, 56]])

In [18]:
get_clf_eval(y_test, custom_proba)

Confusion Matrix
[[86 24]
 [13 56]]
********************
accuracy precision recall
0.7932960893854749 0.7 0.8115942028985508
